# Ovillos — Data Cleaning & Normalization

Pipeline de limpieza para el catálogo de ovillos extraído del SIC JAC.

## Flujo del pipeline

| Paso | Qué se hace | Output |
|------|------------|--------|
| 1 | Cargar y explorar datos fuente | Overview del DataFrame |
| 2 | Limpiar `codigo` → extraer `tipo`, `estado_mat`, `descripcion_material` | `df-code-cleaning.csv` |
| 3 | Extraer `titulo` (grosor/presentación) desde `descripcion` | `titulo` poblado |
| 4 | Separar `descripcion` en `color` + `color_id` | `color`, `color_id` poblados |
| 5 | Edge cases: duplicados, MIXTO, TIPO-CH, residuales en descripcion | Limpieza aplicada |
| 6 | Exportar dataset limpio final | *(pendiente)* |

**Fuente:** `docs/reference/warehouse/cleaned/ovillos-prod.xlsx`

# 1. Carga y exploración inicial

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

SOURCE = Path("../../reference/warehouse/cleaned/ovillos-prod.xlsx")
OUTPUT_DIR = Path("../../reference/warehouse/cleaned")

In [2]:
df = pd.read_excel(SOURCE)
print(f"Shape: {df.shape}")
df.head(10)

Shape: (2132, 12)


,codigo,descripcion,unidad,Saldoin-Cantidad,Saldoin-Valor,ent-Cantidad,ent-Valor,sal-Cantidad,sal-Valor,sadout-Cantidad,saldout-Valor,SUB-CATEGORIA
0,01-26-01,KANTUTA 3045 2/18,KILOS,0.0,0.0,203.0,203.0,203.0,203.0,0.0,0.0,2/18
1,01-26-05,AM. LIMON 0024 2/18,KILOS,0.0,0.0,201.5,201.5,201.5,201.5,0.0,0.0,2/18
2,01-26-06,AZUL PETROLEO 7009 2/18,KILOS,0.0,0.0,203.0,203.0,203.0,203.0,0.0,0.0,2/18
3,01-26-11,BLANCO 0010 2/18,KILOS,0.0,0.0,202.5,202.5,202.5,202.5,0.0,0.0,2/18
4,01-26-13,BLANCO 0010 2/18,KILOS,0.0,0.0,199.0,199.0,199.0,199.0,0.0,0.0,2/18
5,01-26-16,V. ESMERALDA 4033 2/18,KILOS,0.0,0.0,203.5,203.5,203.5,203.5,0.0,0.0,2/18
6,01-26-18,AZUL PASTEL 7012 2/18,KILOS,0.0,0.0,203.5,203.5,203.5,203.5,0.0,0.0,2/18
7,01-26-19,VERDE 4073 2/18,KILOS,0.0,0.0,204.0,204.0,204.0,204.0,0.0,0.0,2/18
8,01-26-21,CICLAN 2004 2/18,KILOS,0.0,0.0,204.0,204.0,204.0,204.0,0.0,0.0,2/18
9,01-26-23,CICLAN OSCURO 2005 2/18,KILOS,0.0,0.0,203.0,203.0,203.0,203.0,0.0,0.0,2/18


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2132 entries, 0 to 2131
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   codigo            2132 non-null   str    
 1   descripcion       2132 non-null   str    
 2   unidad            2132 non-null   str    
 3   Saldoin-Cantidad  2132 non-null   float64
 4   Saldoin-Valor     2132 non-null   float64
 5   ent-Cantidad      2132 non-null   float64
 6   ent-Valor         2132 non-null   float64
 7   sal-Cantidad      2132 non-null   float64
 8   sal-Valor         2132 non-null   float64
 9   sadout-Cantidad   2132 non-null   float64
 10  saldout-Valor     2132 non-null   float64
 11  SUB-CATEGORIA     2132 non-null   str    
dtypes: float64(8), str(4)
memory usage: 200.0 KB


In [4]:
print(f"Nulls por columna:\n{df.isnull().sum()}")

Nulls por columna:
codigo              0
descripcion         0
unidad              0
Saldoin-Cantidad    0
Saldoin-Valor       0
ent-Cantidad        0
ent-Valor           0
sal-Cantidad        0
sal-Valor           0
sadout-Cantidad     0
saldout-Valor       0
SUB-CATEGORIA       0
dtype: int64


# 2. Limpieza de `codigo`

## Estructura del código

El `codigo` tiene un identificador base `XX-YY-ZZ` y puede incluir **prefijos** (tipo de material) o **sufijos** (estado/máquina).

**Objetivo:** separar el identificador base en tres columnas nuevas:
- `tipo` — tipo de hilo: HB, N, SM, STOLL, CH, M, CICLAN, ALPACA, AP
- `estado_mat` — estado del material: REC (recuperado) o MAT-REC (material recuperado)
- `descripcion_material` — forma del material: CONO, ALM (almacén)

## 2.1 — Analizar prefijos y sufijos en codigo


In [5]:
codes = df['codigo'].astype(str)

print(f"Códigos únicos: {df['codigo'].nunique()} de {len(df)} rows")
print(f"\nPrefijos:")
print(f"  'REC-': {codes.str.startswith('REC-').sum()}")
print(f"  'MAT-': {codes.str.startswith('MAT-').sum()}")
print(f"  'CC-': {codes.str.startswith('CC-').sum()}")
print(f"  'ALP-', 'ALPACA-', 'ALPACA ': {codes.str.contains('ALP').sum()}")
print(f"\nSufijos:")
print(f"  '-N': {codes.str.endswith('-N').sum()}")
print(f"  '-SM': {codes.str.endswith('-SM').sum()}")
print(f"  '-SN': {codes.str.endswith('-SN').sum()}")
print(f"  '-CH': {codes.str.endswith('-CH').sum()}")
print(f"  '-M': {codes.str.endswith('-M').sum()}")
print(f"  '-STOLL': {codes.str.endswith('-STOLL').sum()}")

print(f"\nCódigos largos (>8 chars): {codes.str.len().gt(8).sum()}")
print(codes[codes.str.len() > 8].unique())

Códigos únicos: 2132 de 2132 rows

Prefijos:
  'REC-': 25
  'MAT-': 14
  'CC-': 11
  'ALP-', 'ALPACA-', 'ALPACA ': 5

Sufijos:
  '-N': 202
  '-SM': 55
  '-SN': 7
  '-CH': 2
  '-M': 3
  '-STOLL': 2

Códigos largos (>8 chars): 454
<StringArray>
[ '09-32-26-01',  '09-32-26-02',  '09-32-26-03',  '09-32-26-04',
  '09-32-26-05',  '09-32-26-06',  '09-32-26-07',  '09-32-26-08',
  '09-32-26-09',  '09-32-26-11',
 ...
  '57-26-01-SM',  '57-26-02-SM',  '57-26-03-SM',  '57-26-04-SM',
  '57-26-05-SM',  '57-26-06-SM',  '57-26-07-SM',  '57-26-08-SM',
  '57-26-09-SM', 'MAT-REC 2/15']
Length: 454, dtype: str


## 2.2 — Crear columnas derivadas y asignar tipo/estado según prefijos y sufijos


In [6]:

df['tipo'] = 'HB'  # default
df['estado_mat'] = ''
df['descripcion_material'] = ''

# --- Asignar tipo según sufijo ---
df.loc[df['codigo'].str.endswith('-N'), 'tipo'] = 'N'
df.loc[df['codigo'].str.endswith('-SM'), 'tipo'] = 'SM'
df.loc[df['codigo'].str.endswith('-M'), 'tipo'] = 'M'
df.loc[df['codigo'].str.endswith('-CH'), 'tipo'] = 'CH'
df.loc[df['codigo'].str.endswith('-STOLL'), 'tipo'] = 'STOLL'

# Combinaciones especiales
df.loc[df['codigo'].str.endswith('-SN'), 'tipo'] = 'SM N' 
df.loc[df['codigo'].str.endswith('-SM-N'), 'tipo'] = 'SM N'

# --- Tipo según prefijo ---
df.loc[df['codigo'].str.startswith('CC-'), 'tipo'] = 'CICLAN'

# --- Estado del material ---
df.loc[df['codigo'].str.startswith('REC-'), 'estado_mat'] = 'REC'
df.loc[df['codigo'].str.contains(r'^MAT-REC[\s-]', regex=True, na=False), 'estado_mat'] = 'MAT-REC'

# --- Casos ALPACA ---
df.loc[df['codigo'].str.startswith(('ALP-', 'ALPACA-')), 'tipo'] = 'ALPACA'
df.loc[df['codigo'].str.contains('ALP', regex=False) & df['codigo'].str.endswith('N'), 'tipo'] = 'ALPACA / SM N'

# Verificar distribución
print("Distribución de tipo:")
print(df['tipo'].value_counts())

Distribución de tipo:
tipo
HB               1848
N                 201
SM                 55
CICLAN             11
SM N                7
M                   3
STOLL               2
ALPACA              2
CH                  2
ALPACA / SM N       1
Name: count, dtype: int64


## 2.3 — Limpiar sufijos y prefijos del codigo (dejar solo el identificador base)


In [7]:

SUFFIXES = ['-SM', '-SM-N', '-SN', '-CH', '-N', '-M', '-STOLL']
PREFIXES = ['REC-', 'MAT-REC-', 'CC-', 'MAT-REC ', 'ALP-', 'ALPACA-', 'ALPACA ']

for suffix in SUFFIXES:
    mask = df['codigo'].str.endswith(suffix)
    df.loc[mask, 'codigo'] = df.loc[mask, 'codigo'].str.replace(re.escape(suffix) + '$', '', regex=True)

for prefix in PREFIXES:
    mask = df['codigo'].str.startswith(prefix)
    df.loc[mask, 'codigo'] = df.loc[mask, 'codigo'].str.replace('^' + re.escape(prefix), '', regex=True)

print(f"Códigos únicos post-limpieza: {df['codigo'].nunique()} de {len(df)} rows")

Códigos únicos post-limpieza: 2055 de 2132 rows


## 2.4 — Códigos con letras residuales (casos especiales: REC, MAT-REC, CONO, ALM, AP)


In [8]:

with_words = df['codigo'].str.contains(r'[A-Za-z]', regex=True, na=False)
print(f"Filas con letras en codigo post-limpieza: {with_words.sum()}")
df[with_words][['codigo']].head(10)

Filas con letras en codigo post-limpieza: 7


,codigo
1339,20-17-REC
1378,3/15-CONO
1379,MAT-REC
1401,ALM-1
1402,ALM-1
2101,19-24-05-SM-1
2102,AP-23-02


## 2.5 — Resolver casos especiales residuales


In [9]:

# Asignar estado_mat para códigos que terminan en -REC
df.loc[df['codigo'].str.endswith('-REC'), 'estado_mat'] = 'REC'

# Asignar descripcion_material
df.loc[df['codigo'].str.endswith('-CONO'), 'descripcion_material'] = 'CONO'
df.loc[df['codigo'].str.startswith('ALM-'), 'descripcion_material'] = 'ALM'

# Asignar tipo para casos con -SM- (SM en medio del código)
df.loc[df['codigo'].str.contains('-SM-', regex=False), 'tipo'] = 'SM'

# Asignar tipo AP
df.loc[df['codigo'].str.startswith('AP-'), 'tipo'] = 'AP'

# Limpiar prefijos/sufijos restantes
df.loc[df['codigo'].str.endswith('MAT-REC'), 'estado_mat'] = 'MAT-REC'
df.loc[df['codigo'].str.endswith('MAT-REC'), 'codigo'] = 'N/A'

for prefix in ['ALM-', 'AP-']:
    mask = df['codigo'].str.startswith(prefix)
    df.loc[mask, 'codigo'] = df.loc[mask, 'codigo'].str.replace('^' + prefix, '', regex=True)

for suffix in ['-CONO', '-REC']:
    mask = df['codigo'].str.endswith(suffix)
    df.loc[mask, 'codigo'] = df.loc[mask, 'codigo'].str.replace(re.escape(suffix) + '$', '', regex=True)

# Limpiar -SM residual (cuando está en medio del código)
df.loc[df['codigo'].str.contains('-SM', regex=False), 'codigo'] = (
    df.loc[df['codigo'].str.contains('-SM', regex=False), 'codigo'].str.replace('-SM', '', regex=True)
)

## 2.6 — Verificar resultado final del codigo


In [10]:

with_words = df['codigo'].str.contains(r'[A-Za-z]', regex=True, na=False)
print(f"Códigos con letras residuales: {with_words.sum()}")
print(f"Códigos únicos: {df['codigo'].nunique()} de {len(df)} rows")
print(f"Duplicados: {df['codigo'].duplicated().sum()}")
print(f"\nValores únicos en tipo:\n{df['tipo'].unique()}")
print(f"\nValores únicos en estado_mat:\n{df['estado_mat'].unique()}")
print(f"\nValores únicos en descripcion_material:\n{df['descripcion_material'].unique()}")

Códigos con letras residuales: 1
Códigos únicos: 2054 de 2132 rows
Duplicados: 78

Valores únicos en tipo:
<StringArray>
[           'HB',        'CICLAN',             'N',            'SM',
         'STOLL', 'ALPACA / SM N',        'ALPACA',            'CH',
             'M',          'SM N',            'AP']
Length: 11, dtype: str

Valores únicos en estado_mat:
<StringArray>
['', 'REC', 'MAT-REC']
Length: 3, dtype: str

Valores únicos en descripcion_material:
<StringArray>
['', 'CONO', 'ALM']
Length: 3, dtype: str


## 2.7 — Exportar resultados intermedios


In [11]:

df.to_csv(OUTPUT_DIR / 'df-code-cleaning.csv', index=False)
print(f"Exportado: {OUTPUT_DIR / 'df-code-cleaning.csv'}")

Exportado: ../../reference/warehouse/cleaned/df-code-cleaning.csv


# 3. Extracción de `titulo` desde `descripcion`

La columna `descripcion` tiene el formato: `COLOR  CODIGO_COLOR  GROSOR`

Ejemplos:
- `KANTUTA  3045  2/18` → color=KANTUTA, codigo_color=3045, grosor=2/18
- `AM. LIMON  0024  2/18` → color=AM. LIMON, codigo_color=0024, grosor=2/18

**Objetivo:** extraer `titulo` (grosor o presentación) de la descripción,
dejando `descripcion` solo con color + código de color.

## 3.1 — Estrategia de extracción

Se aplican tres estrategias en orden:
1. **SUB-CATEGORIA**: si el grosor ya está en `SUB-CATEGORIA`, se copia directo
2. **Patrón X/XX**: si no hay título, se busca el primer patrón `\d+/\d+` en `descripcion`
3. **Casos especiales**: RECUPERADO, STOLL, MIXTO, TIPO-CH

In [12]:
dframe = pd.read_csv(OUTPUT_DIR / 'df-code-cleaning.csv')
print(f"Cargado: {dframe.shape}")

Cargado: (2132, 15)


## 3.2 — Estrategia 1: copiar SUB-CATEGORIA cuando aparece en descripcion


In [13]:

dframe['titulo'] = ''

mask_in_desc = dframe.apply(
    lambda row: str(row['SUB-CATEGORIA']) in str(row['descripcion']), axis=1
)

dframe.loc[mask_in_desc, 'titulo'] = dframe.loc[mask_in_desc, 'SUB-CATEGORIA']
dframe.loc[mask_in_desc, 'descripcion'] = dframe.loc[mask_in_desc].apply(
    lambda row: str(row['descripcion']).replace(str(row['SUB-CATEGORIA']), '').strip(), axis=1
)

print(f"Estrategia 1 (SUB-CATEGORIA en descripcion): {mask_in_desc.sum()} / {len(dframe)}")

Estrategia 1 (SUB-CATEGORIA en descripcion): 1811 / 2132


## 3.3 — Estrategia 2: buscar patrón X/XX para los que quedan sin titulo

In [14]:

sin_titulo = dframe['titulo'] == ''

thickness = dframe.loc[sin_titulo, 'descripcion'].str.extract(r'(\d+/\d+)', expand=False)

dframe.loc[sin_titulo & thickness.notna(), 'titulo'] = thickness
dframe.loc[sin_titulo & thickness.notna(), 'descripcion'] = (
    dframe.loc[sin_titulo & thickness.notna(), 'descripcion']
    .str.replace(r'\s*\d+/\d+\s*', ' ', regex=True)
    .str.strip()
)

print(f"Estrategia 2 (patrón X/XX): {thickness.notna().sum()} asignados")
print(f"Total con titulo: {(dframe['titulo'] != '').sum()} / {len(dframe)}")
print(f"Aún sin titulo: {(dframe['titulo'] == '').sum()} / {len(dframe)}")

Estrategia 2 (patrón X/XX): 305 asignados
Total con titulo: 2116 / 2132
Aún sin titulo: 16 / 2132


## 3.4 — Distribución de titulos obtenidos

In [15]:

dframe.groupby(['titulo']).size().reset_index(name='counts').sort_values(by='counts', ascending=False)

,titulo,counts
10,2/18,875
17,2/32,472
4,2/12,256
5,2/13,199
19,4/9,83
2,2/10,60
21,STOLL,46
18,3/15,28
3,2/11,24
7,2/15,19


## 3.5 — Casos RECUPERADO

Filas donde `SUB-CATEGORIA = RECUPERADO`. Estas filas tienen `descripcion` como:
- `MATERIAL  X/XX`
- `MATERIAL  X/XX STOLL`
- `SAL ANT PROD TERM`

Se extrae el grosor `X/XX` y se limpia la palabra "MATERIAL".

In [16]:
rec_mask = dframe['titulo'] == 'RECUPERADO'
print(f"Filas RECUPERADO: {rec_mask.sum()}")
dframe.loc[rec_mask]

Filas RECUPERADO: 14


,codigo,descripcion,unidad,Saldoin-Cantidad,Saldoin-Valor,ent-Cantidad,ent-Valor,sal-Cantidad,sal-Valor,sadout-Cantidad,saldout-Valor,SUB-CATEGORIA,tipo,estado_mat,descripcion_material,titulo
1339,20-17,SAL ANT PROD TERM,KILOS,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,RECUPERADO,HB,REC,NaN,RECUPERADO
1367,2/10,MATERIAL 2/10,KILOS,26.0,26.0,522.0,522.0,170.0,170.0,378.0,378.0,RECUPERADO,HB,MAT-REC,NaN,RECUPERADO
1368,2/11,MATERIAL 2/11 SM,KILOS,58.0,58.0,57.0,57.0,0.0,0.0,115.0,115.0,RECUPERADO,HB,MAT-REC,NaN,RECUPERADO
1369,2/12,MATERIAL 2/12,KILOS,82.0,82.0,1649.0,1649.0,395.0,395.0,1336.0,1336.0,RECUPERADO,HB,MAT-REC,NaN,RECUPERADO
1370,2/13,MATERIAL 2/13,KILOS,542.0,542.0,1055.0,1055.0,0.0,0.0,1597.0,1597.0,RECUPERADO,HB,MAT-REC,NaN,RECUPERADO
1371,2/14,MATERIAL 2/14,KILOS,9.0,9.0,0.0,0.0,0.0,0.0,9.0,9.0,RECUPERADO,HB,MAT-REC,NaN,RECUPERADO
1372,2/14,MATERIAL 2/14 STOLL,KILOS,228.0,228.0,0.0,0.0,9.0,9.0,219.0,219.0,RECUPERADO,STOLL,MAT-REC,NaN,RECUPERADO
1373,2/18,MATERIAL 2/18,KILOS,307.0,307.0,1914.0,1914.0,1945.0,1945.0,276.0,276.0,RECUPERADO,HB,MAT-REC,NaN,RECUPERADO
1374,2/28,MATERIAL 2/28,KILOS,115.0,115.0,0.0,0.0,0.0,0.0,115.0,115.0,RECUPERADO,HB,MAT-REC,NaN,RECUPERADO
1375,2/30,MATERIAL 2/30,KILOS,77.0,77.0,3.0,3.0,0.0,0.0,80.0,80.0,RECUPERADO,HB,MAT-REC,NaN,RECUPERADO


In [17]:
# Extraer X/XX de las RECUPERADO y limpiar "MATERIAL"

thick_rec = dframe.loc[rec_mask, 'descripcion'].str.extract(r'(\d+/\d+)', expand=False)
dframe.loc[rec_mask & thick_rec.notna(), 'titulo'] = thick_rec

# Quitar el grosor de descripcion
dframe.loc[rec_mask & thick_rec.notna(), 'descripcion'] = (
    dframe.loc[rec_mask & thick_rec.notna(), 'descripcion']
    .str.replace(r'\s*\d+/\d+\s*', ' ', regex=True)
    .str.strip()
)

# Quitar "MATERIAL"
dframe.loc[rec_mask, 'descripcion'] = (
    dframe.loc[rec_mask, 'descripcion']
    .str.replace(r'MATERIAL\s*', '', regex=True)
    .str.strip()
)

print("RECUPERADO después de procesar:")
dframe.loc[rec_mask][['codigo', 'descripcion', 'titulo']]

RECUPERADO después de procesar:


,codigo,descripcion,titulo
1339,20-17,SAL ANT PROD TERM,RECUPERADO
1367,2/10,,2/10
1368,2/11,SM,2/11
1369,2/12,,2/12
1370,2/13,,2/13
1371,2/14,,2/14
1372,2/14,STOLL,2/14
1373,2/18,,2/18
1374,2/28,,2/28
1375,2/30,,2/30


## 3.6 — Casos STOLL

Filas donde `SUB-CATEGORIA = STOLL` (hilo para máquina Stoll). 
Estos registros tienen el grosor ya en `SUB-CATEGORIA`, por lo que el `titulo` queda
como `STOLL` inicialmente. Se mueve ese valor a la columna `tipo` y se deja `titulo` vacío.

In [18]:
stoll_mask = dframe['titulo'] == 'STOLL'
print(f"Filas STOLL: {stoll_mask.sum()}")

# Asignar tipo = STOLL para estas filas
dframe.loc[stoll_mask, 'tipo'] = 'STOLL'
dframe.loc[stoll_mask, 'titulo'] = ''

dframe.loc[dframe['SUB-CATEGORIA'] == 'STOLL'][['codigo', 'descripcion', 'tipo', 'titulo', 'SUB-CATEGORIA']].head(10)

Filas STOLL: 46


,codigo,descripcion,tipo,titulo,SUB-CATEGORIA
2051,01-26-29,BLANCO 0010,STOLL,,STOLL
2052,02-26-53,NEGRO 5000,STOLL,,STOLL
2053,03-26-10,AZUL 7052,STOLL,,STOLL
2054,03-26-22,AZUL 7052,STOLL,,STOLL
2055,05-26-13,VERDE OSC. 4016,STOLL,,STOLL
2056,05-26-27,VERDE OSC. 4016,STOLL,,STOLL
2057,05-26-36,AZUL 7052,STOLL,,STOLL
2058,05-26-41,NEGRO 5000,STOLL,,STOLL
2059,07-26-10,NEGRO 5000,STOLL,,STOLL
2060,07-26-15,API 7033,STOLL,,STOLL


## 3.7 — Limpieza de valores residuales en `titulo`

**Regla A:** si `titulo = 'RECUPERADO'` y `SUB-CATEGORIA = 'RECUPERADO'` → vaciar `titulo` (redundante).

**Regla B:** si `titulo` contiene `'SN'` (ej: `'2/28 SN'`), extraer `'SN'` y:
- Si `tipo = 'HB'` → `tipo = 'SM N'` y limpiar `'SN'` del titulo
- Si `tipo != 'HB'` → imprimir el valor actual para revisión

### 3.7a — Regla A: RECUPERADO redundante

In [19]:

rec_ambos = (dframe['titulo'] == 'RECUPERADO') & (dframe['SUB-CATEGORIA'] == 'RECUPERADO')
print(f"Filas con RECUPERADO en titulo y SUB-CATEGORIA: {rec_ambos.sum()}")

if rec_ambos.any():
    print(dframe.loc[rec_ambos, ['codigo', 'descripcion', 'SUB-CATEGORIA', 'tipo', 'titulo']].to_string())
    dframe.loc[rec_ambos, 'titulo'] = ''
    print("→ titulo limpiado (vacío)")

Filas con RECUPERADO en titulo y SUB-CATEGORIA: 2
     codigo        descripcion SUB-CATEGORIA   tipo      titulo
1339  20-17  SAL ANT PROD TERM    RECUPERADO     HB  RECUPERADO
1379    NaN              STOLL    RECUPERADO  STOLL  RECUPERADO
→ titulo limpiado (vacío)


### 3.7b — Regla B: 'SN' en titulo


In [20]:
# Usar str.replace vectorizado de pandas (maneja NaN sin errores de tipo)

sn_mask = dframe['titulo'].str.contains(r'\bSN\b', regex=True, na=False)
print(f"Filas con 'SN' en titulo: {sn_mask.sum()}")

if sn_mask.any():
    # Limpiar 'SN' del titulo (vectorizado)
    dframe.loc[sn_mask, 'titulo'] = (
        dframe.loc[sn_mask, 'titulo']
        .str.replace(r'\s*SN\s*', '', regex=True)
        .str.strip()
    )

    # Revisar tipo para cada fila
    for idx in dframe.index[sn_mask]:
        tipo_actual = dframe.loc[idx, 'tipo']
        if tipo_actual == 'HB':
            dframe.loc[idx, 'tipo'] = 'SM N'
            print(f"  {idx}: HB → SM N")
        else:
            print(f"  ATENCIÓN {idx}: tipo={tipo_actual} (NO es HB). titulo limpiado, tipo intacto.")

Filas con 'SN' en titulo: 2
  ATENCIÓN 2045: tipo=SM N (NO es HB). titulo limpiado, tipo intacto.
  ATENCIÓN 2047: tipo=SM N (NO es HB). titulo limpiado, tipo intacto.


## 3.8 — Resumen de titulo post-procesamiento


In [21]:

print(f"Filas con titulo: {(dframe['titulo'].notna() & (dframe['titulo'] != '')).sum()} / {len(dframe)}")
print(f"Filas sin titulo: {(dframe['titulo'].isna() | (dframe['titulo'] == '')).sum()} / {len(dframe)}")
print()
dframe.groupby(['titulo']).size().reset_index(name='counts').sort_values(by='counts', ascending=False)

Filas con titulo: 2068 / 2132
Filas sin titulo: 64 / 2132



,titulo,counts
10,2/18,876
16,2/32,473
4,2/12,257
5,2/13,200
18,4/9,83
0,,64
2,2/10,61
17,3/15,30
3,2/11,25
7,2/15,19


## 3.9 — Exportar resultados intermedios


In [22]:

dframe.to_csv(OUTPUT_DIR / 'df-yarn-cleaning.csv', index=False)
print(f"Exportado: {OUTPUT_DIR / 'df-yarn-cleaning.csv'}")

Exportado: ../../reference/warehouse/cleaned/df-yarn-cleaning.csv


# 4. Limpiar `descripcion` — extraer estado_mat

La columna `descripcion` tiene valores de estado que deben extraerse a `estado_mat`:

| Valor en descripcion | estado_mat | Acción sobre descripción |
|---|---|---|
| `MATERIAL RECUPERADO` | `MAT-REC` | Eliminar el texto |
| `RECUPERADO` | `REC` | Eliminar el texto, dejar solo el color |
| `(REMATE)` | `REMATE` | Eliminar `REMATE` y `()` |
| `SM` (exacto) | `SM N` | Dejar vacío |
| `STOLL` (exacto) | `STOLL` | Dejar vacío |

Después de la limpieza, se listan los otros valores no-numéricos que quedan en `descripcion`.


## 4.1 — Cargar datos intermedios


In [23]:
dframe = pd.read_csv(OUTPUT_DIR / 'df-yarn-cleaning.csv', dtype={'descripcion_material': object})
print(f"Total filas: {len(dframe)}  |  Columnas: {list(dframe.columns)}")
dframe['estado_mat'].value_counts(dropna=False).reset_index(name='counts')

Total filas: 2132  |  Columnas: ['codigo', 'descripcion', 'unidad', 'Saldoin-Cantidad', 'Saldoin-Valor', 'ent-Cantidad', 'ent-Valor', 'sal-Cantidad', 'sal-Valor', 'sadout-Cantidad', 'saldout-Valor', 'SUB-CATEGORIA', 'tipo', 'estado_mat', 'descripcion_material', 'titulo']


,estado_mat,counts
0,NaN,2092
1,REC,26
2,MAT-REC,14


## 4.2 — MATERIAL RECUPERADO → `MAT-REC`

Si `descripcion` contiene 'MATERIAL RECUPERADO', se extrae a `estado_mat = 'MAT-REC'` y se limpia de `descripcion`.


In [24]:
mat_rec_mask = dframe['descripcion'].str.contains(r'MATERIAL\s+RECUPERADO', regex=True, na=False)
print(f"MATERIAL RECUPERADO en descripcion: {mat_rec_mask.sum()}")

if mat_rec_mask.any():
    dframe.loc[mat_rec_mask, 'descripcion'] = (
        dframe.loc[mat_rec_mask, 'descripcion']
        .str.replace(r'\s*MATERIAL\s+RECUPERADO\s*', '', regex=True)
        .str.strip()
    )
    dframe.loc[mat_rec_mask, 'estado_mat'] = 'MAT-REC'

dframe.loc[mat_rec_mask, ['codigo', 'descripcion', 'SUB-CATEGORIA', 'estado_mat', 'tipo']]

MATERIAL RECUPERADO en descripcion: 3


,codigo,descripcion,SUB-CATEGORIA,estado_mat,tipo
1401,1,,MIXTO,MAT-REC,HB
1402,1,,MIXTO,MAT-REC,N
2131,2/15,,2/15,MAT-REC,HB


## 4.3 — RECUPERADO → `REC`

Para las filas RESTANTES (que no matchearon MATERIAL RECUPERADO), si `descripcion` contiene 'RECUPERADO', se extrae a `estado_mat = 'REC'` y se limpia dejando solo el color.


In [25]:
rec_mask = dframe['descripcion'].str.contains(r'\bRECUPERADO\b', regex=True, na=False)
print(f"RECUPERADO en descripcion (restante): {rec_mask.sum()}")

if rec_mask.any():
    dframe.loc[rec_mask, 'descripcion'] = (
        dframe.loc[rec_mask, 'descripcion']
        .str.replace(r'\s*RECUPERADO\s*', '', regex=True)
        .str.strip()
    )
    dframe.loc[rec_mask, 'estado_mat'] = 'REC'

dframe.loc[rec_mask, ['codigo', 'descripcion', 'SUB-CATEGORIA', 'estado_mat', 'tipo']]

RECUPERADO en descripcion (restante): 10


,codigo,descripcion,SUB-CATEGORIA,estado_mat,tipo
1391,23-07,VERDE BANDERA,MIXTO,REC,HB
1392,23-09,BLANCO,MIXTO,REC,HB
1393,23-10,VICUÑA,MIXTO,REC,HB
1394,23-11,BLANCO,MIXTO,REC,HB
1395,23-12,BLANCO,MIXTO,REC,HB
1396,23-13,BLANCO,MIXTO,REC,HB
1397,23-14,BLANCO,MIXTO,REC,HB
1398,23-17,ROJO,MIXTO,REC,HB
1399,23-18,NEGRO,MIXTO,REC,HB
1400,25-01,VICUÑA,MIXTO,REC,HB


## 4.4 — `(REMATE)` → `REMATE`

Si `descripcion` contiene 'REMATE', se extrae a `estado_mat = 'REMATE'` y se elimina junto con los paréntesis de `descripcion`.


In [26]:
remate_mask = dframe['descripcion'].str.contains(r'\bREMATE\b', regex=True, na=False)
print(f"REMATE en descripcion: {remate_mask.sum()}")

if remate_mask.any():
    for idx in dframe.index[remate_mask]:
        dframe.loc[idx, 'estado_mat'] = 'REMATE'
    dframe.loc[remate_mask, 'descripcion'] = (
        dframe.loc[remate_mask, 'descripcion']
        .str.replace(r'\s*\(?\s*REMATE\s*\)?\s*', '', regex=True)
        .str.strip()
    )

dframe.loc[remate_mask, ['codigo', 'descripcion', 'SUB-CATEGORIA', 'estado_mat', 'tipo']]

REMATE en descripcion: 1


,codigo,descripcion,SUB-CATEGORIA,estado_mat,tipo
1352,25-06-25-22,GRISS,RECUPERADO,REMATE,HB


## 4.5 — `SM` → `SM`

Si `descripcion` es exactamente 'SM', se extrae a `estado_mat = 'SM N'` y se deja vacía.


In [27]:
sm_mask = dframe['descripcion'].str.fullmatch(r'SM', na=False)
print(f"SM exacto en descripcion: {sm_mask.sum()}")
if sm_mask.any():
    dframe.loc[sm_mask, 'tipo'] = 'SM'
    dframe.loc[sm_mask, 'descripcion'] = ''

dframe.loc[sm_mask, ['codigo', 'descripcion', 'SUB-CATEGORIA', 'estado_mat', 'tipo']]

SM exacto en descripcion: 1


,codigo,descripcion,SUB-CATEGORIA,estado_mat,tipo
1368,2/11,,RECUPERADO,MAT-REC,SM


## 4.6 — `STOLL` → `STOLL`

Si `descripcion` es exactamente 'STOLL', se extrae a `estado_mat = 'STOLL'` y se deja vacía.


In [28]:
stoll_mask = dframe['descripcion'].str.fullmatch(r'STOLL', na=False)
print(f"STOLL exacto en descripcion: {stoll_mask.sum()}")
if stoll_mask.any():
    dframe.loc[stoll_mask, 'tipo'] = 'STOLL'
    dframe.loc[stoll_mask, 'descripcion'] = ''

dframe.loc[stoll_mask, ['codigo', 'descripcion', 'SUB-CATEGORIA', 'estado_mat', 'tipo']]

STOLL exacto en descripcion: 2


,codigo,descripcion,SUB-CATEGORIA,estado_mat,tipo
1372,2/14,,RECUPERADO,MAT-REC,STOLL
1379,NaN,,RECUPERADO,MAT-REC,STOLL


## 4.7 — Otros valores no-numéricos en `descripcion`

Muestra los valores de `descripcion` que quedan sin números (no son color + código) después de la limpieza.


In [29]:
sin_numeros = ~dframe['descripcion'].str.contains(r'\d', regex=True, na=False)

# Filas con descripcion sin numeros (detalle)
dframe.loc[sin_numeros & dframe['descripcion'].notna(),
           ['codigo', 'descripcion', 'SUB-CATEGORIA', 'estado_mat', 'tipo', 'titulo']]\
    .sort_values('descripcion')

,codigo,descripcion,SUB-CATEGORIA,estado_mat,tipo,titulo
2131,2/15,,2/15,MAT-REC,HB,2/15
1401,1,,MIXTO,MAT-REC,HB,NaN
1368,2/11,,RECUPERADO,MAT-REC,SM,2/11
1372,2/14,,RECUPERADO,MAT-REC,STOLL,2/14
1379,NaN,,RECUPERADO,MAT-REC,STOLL,NaN
1402,1,,MIXTO,MAT-REC,N,NaN
1392,23-09,BLANCO,MIXTO,REC,HB,NaN
1394,23-11,BLANCO,MIXTO,REC,HB,NaN
1395,23-12,BLANCO,MIXTO,REC,HB,NaN
1396,23-13,BLANCO,MIXTO,REC,HB,NaN


## 4.8 — `SAL ANT PROD TERM` → `descripcion_material`

'SAL ANT PROD TERM' es un valor que describe el tipo de material, no un color. Se extrae a la nueva columna `descripcion_material`.


In [30]:
other_desc_mask = dframe['descripcion'].str.fullmatch(r'SAL ANT PROD TERM', na=False)
print(f"SAL ANT PROD TERM exacto en descripcion: {other_desc_mask.sum()}")
if other_desc_mask.any():
    dframe.loc[other_desc_mask, 'descripcion_material'] = 'SAL ANT PROD TERM'
    dframe.loc[other_desc_mask, 'descripcion'] = ''

dframe.loc[other_desc_mask, ['codigo', 'descripcion', 'SUB-CATEGORIA', 'estado_mat', 'tipo', 'descripcion_material']]

SAL ANT PROD TERM exacto en descripcion: 2


,codigo,descripcion,SUB-CATEGORIA,estado_mat,tipo,descripcion_material
1092,20-17-2/32,,2/32,NaN,HB,SAL ANT PROD TERM
1339,20-17,,RECUPERADO,REC,HB,SAL ANT PROD TERM


## 4.9 — `CONO` → `descripcion_material`

'CONO' indica el formato del ovillo, no un color. Se extrae a `descripcion_material`.


In [31]:
cono_mask = dframe['descripcion'].str.fullmatch(r'CONO', na=False)
print(f"CONO exacto en descripcion: {cono_mask.sum()}")
if cono_mask.any():
    dframe.loc[cono_mask, 'descripcion_material'] = 'CONO'
    dframe.loc[cono_mask, 'descripcion'] = ''

dframe.loc[cono_mask, ['codigo', 'descripcion', 'SUB-CATEGORIA', 'estado_mat', 'tipo', 'descripcion_material']]

CONO exacto en descripcion: 1


,codigo,descripcion,SUB-CATEGORIA,estado_mat,tipo,descripcion_material
1378,3/15,,RECUPERADO,MAT-REC,HB,CONO


## 4.10 — Sufijos en `descripcion` (`-N`, `-D`, etc.)

Algunas descripciones tienen sufijos con guión al final:

| Sufijo | Filas | Acción |
|---|---|---|
| `-N` | 18 | Extraer a `tipo`, limpiar descripcion |
| `-D` | 127 | Agregar a `descripcion_material`, limpiar descripcion |


In [32]:
# Extraer sufijo -N → tipo='N'
suf_n = dframe['descripcion'].str.contains(r'-N$', regex=True, na=False)
print(f"Sufijo -N en descripcion: {suf_n.sum()}")

if suf_n.any():
    dframe.loc[suf_n, 'tipo'] = 'N'
    dframe.loc[suf_n, 'descripcion'] = (
        dframe.loc[suf_n, 'descripcion']
        .str.replace(r'\s*-N\s*$', '', regex=True)
        .str.strip()
    )

dframe.loc[suf_n, ['codigo', 'descripcion', 'SUB-CATEGORIA', 'estado_mat', 'tipo']]

Sufijo -N en descripcion: 18


,codigo,descripcion,SUB-CATEGORIA,estado_mat,tipo
1527,03-26-16,BLANCO 0010,2/13 N,NaN,N
1528,03-26-17,PERLA 0031,2/13 N,NaN,N
1529,03-26-18,NEGRO 5000,2/13 N,NaN,N
1530,11-25-17,ARENA 6001,2/13 N,NaN,N
1548,17-25-09,BEIGE 6039,2/13 N,NaN,N
1556,20-26-11,PACEÑITA 2010,2/13 N,NaN,N
1558,20-26-13,BLANCO 0010,2/13 N,NaN,N
1569,21-26-09,CAFE OSCURO 6014,2/13 N,NaN,N
1571,21-26-12,PACEÑITA 2010,2/13 N,NaN,N
1628,36-25-10,MANZANILLA 4081,2/13 N,NaN,N


In [33]:
# Sufijo -D → agregar a descripcion_material y limpiar descripcion
suf_d = dframe['descripcion'].str.contains(r'-D$', regex=True, na=False)
print(f"Sufijo -D en descripcion: {suf_d.sum()} filas")

if suf_d.any():
    # Append -D a descripcion_material (vectorizado)
    has_existing = dframe.loc[suf_d, 'descripcion_material'].notna() & (dframe.loc[suf_d, 'descripcion_material'] != '')
    dframe.loc[suf_d & has_existing, 'descripcion_material'] = dframe.loc[suf_d & has_existing, 'descripcion_material'] + ' - D'
    dframe.loc[suf_d & ~has_existing, 'descripcion_material'] = 'D'

    # Limpiar -D de descripcion
    dframe.loc[suf_d, 'descripcion'] = (
        dframe.loc[suf_d, 'descripcion']
        .str.replace(r'\s*-D\s*$', '', regex=True)
        .str.strip()
    )

dframe.loc[suf_d, ['codigo', 'descripcion', 'SUB-CATEGORIA', 'estado_mat', 'tipo', 'descripcion_material']].head(10)

Sufijo -D en descripcion: 127 filas


,codigo,descripcion,SUB-CATEGORIA,estado_mat,tipo,descripcion_material
850,01-25-06,GUINDO POT. 3035,2/32,NaN,HB,D
851,01-25-35,VERDE BANDERA 4010,2/32,NaN,HB,D
852,01-25-58,ROJO 3006,2/32,NaN,HB,D
877,02-26-37,AZUL MARINO 7013,2/32,NaN,HB,D
903,05-25-50,ROJO 3006,2/32,NaN,HB,D
933,08-25-21,VICUÑA 6007,2/32,NaN,HB,D
1005,13-24-51,VICUÑA 6007,2/32,NaN,HB,D
1006,13-24-55,ROJO 3006,2/32,NaN,HB,D
1019,14-24-25,VICUÑA 6012,2/32,NaN,HB,D
1020,14-24-35,BEIGE 6002,2/32,NaN,HB,D


## 4.11 — Extraer `color` y `color_id` desde `descripcion`

Se extrae el nombre del color y su código numérico a nuevas columnas, y se eliminan de `descripcion`.

Patrón: `COLOR  CÓDIGO` (ej: `KANTUTA  3045`, `AZUL PETROLEO  7009`).

| Columna | Descripción | Ejemplo |
|---|---|---|
| `color` | Nombre del color | `KANTUTA` |
| `color_id` | Código numérico (3-5 dígitos) | `3045` |

Las filas sin código numérico (colores RECUPERADO, etc.) mantienen su `descripcion` intacta.


In [34]:
# Crear columnas vacias
dframe['color'] = None
dframe['color_id'] = None

# Extraer color y color_id desde descripcion (3+ digitos consecutivos)
color_match = dframe['descripcion'].str.extract(r'^(.+?)\s+(\d{3,})', expand=False)
color = color_match[0]
color_id = color_match[1]

# Asignar solo donde hay match
m = color.notna() & color_id.notna()
dframe.loc[m, 'color'] = color[m]
dframe.loc[m, 'color_id'] = color_id[m]

# Eliminar color+color_id de descripcion (replace sobre toda la columna)
dframe['descripcion'] = (
    dframe['descripcion']
    .replace(r'^.+?\s+\d{3,}', '', regex=True)
    .replace(r'^\s+|\s+$', '', regex=True)
)

print(f"Total filas con color+color_id: {m.sum()} / {len(dframe)}")
print(f"Filas sin color_id: {len(dframe) - m.sum()}")
print()
print("Ejemplos de extraccion:")
dframe.loc[m, ['codigo', 'descripcion', 'color', 'color_id']].head(10)

Total filas con color+color_id: 2101 / 2132
Filas sin color_id: 31

Ejemplos de extraccion:


,codigo,descripcion,color,color_id
0,01-26-01,,KANTUTA,3045
1,01-26-05,,AM. LIMON,0024
2,01-26-06,,AZUL PETROLEO,7009
3,01-26-11,,BLANCO,0010
4,01-26-13,,BLANCO,0010
5,01-26-16,,V. ESMERALDA,4033
6,01-26-18,,AZUL PASTEL,7012
7,01-26-19,,VERDE,4073
8,01-26-21,,CICLAN,2004
9,01-26-23,,CICLAN OSCURO,2005


### Filas sin `color_id`

Muestra los valores de `descripcion` que no tienen código numérico.


In [35]:
# Filas que NO tienen color_id
sin_id = dframe['descripcion'].notna() & (dframe['descripcion'] != '') & dframe['color_id'].isna()
print(f"Filas sin color_id: {sin_id.sum()}")

if sin_id.any():
    dframe.loc[sin_id, ['codigo', 'descripcion', 'SUB-CATEGORIA', 'estado_mat', 'tipo', 'color', 'color_id']]

Filas sin color_id: 13


#### Resumen estado_mat post-limpieza


In [36]:
dframe['estado_mat'].value_counts(dropna=False).reset_index(name='counts')

,estado_mat,counts
0,NaN,2091
1,REC,24
2,MAT-REC,16
3,REMATE,1


In [37]:
dframe.info()

<class 'pandas.DataFrame'>
RangeIndex: 2132 entries, 0 to 2131
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   codigo                2131 non-null   str    
 1   descripcion           2123 non-null   str    
 2   unidad                2132 non-null   str    
 3   Saldoin-Cantidad      2132 non-null   float64
 4   Saldoin-Valor         2132 non-null   float64
 5   ent-Cantidad          2132 non-null   float64
 6   ent-Valor             2132 non-null   float64
 7   sal-Cantidad          2132 non-null   float64
 8   sal-Valor             2132 non-null   float64
 9   sadout-Cantidad       2132 non-null   float64
 10  saldout-Valor         2132 non-null   float64
 11  SUB-CATEGORIA         2132 non-null   str    
 12  tipo                  2132 non-null   str    
 13  estado_mat            41 non-null     str    
 14  descripcion_material  132 non-null    object 
 15  titulo                2068 non-n

In [38]:
dframe.to_csv(OUTPUT_DIR / 'df-colour-cleaning.csv', index=False)
print(f"Exportado: {OUTPUT_DIR / 'df-colour-cleaning.csv'}")

Exportado: ../../reference/warehouse/cleaned/df-colour-cleaning.csv


# 5. Edge cases y cleanup


## 5.1 — Análisis de edge cases


In [39]:
# Duplicados en codigo (78 códigos)
dups = dframe[dframe['codigo'].duplicated(keep=False)]\
    .sort_values('codigo')
print(f"Duplicados: {dups.shape[0]} filas, {dframe['codigo'].duplicated().sum()} códigos repetidos")
display(dups[['codigo', 'descripcion', 'SUB-CATEGORIA', 'tipo']].head(20))

print()

# MIXTO (12 filas)
mixto = dframe[dframe['SUB-CATEGORIA'] == 'MIXTO']
print(f"Filas MIXTO: {len(mixto)}")
display(mixto[['codigo', 'descripcion', 'tipo', 'estado_mat', 'titulo']])

print()

# TIPO-CH (3 filas)
ch = dframe[dframe['SUB-CATEGORIA'] == 'TIPO-CH']
print(f"Filas TIPO-CH: {len(ch)}")
ch[['codigo', 'descripcion', 'tipo', 'titulo']]

Duplicados: 154 filas, 78 códigos repetidos


,codigo,descripcion,SUB-CATEGORIA,tipo
1517,03-26-01,,2/13 N,N
884,03-26-01,,2/32,HB
1726,03-26-02,,2/12,HB
1518,03-26-02,,2/13 N,N
1519,03-26-03,,2/13 N,N
82,03-26-03,,2/18,HB
1520,03-26-04,,2/13 N,N
885,03-26-04,,2/32,HB
1418,03-26-05,MADEJITAS,3/15-MADEJITAS,HB
1521,03-26-05,,2/13 N,N



Filas MIXTO: 12


,codigo,descripcion,tipo,estado_mat,titulo
1391,23-07,VERDE BANDERA,HB,REC,NaN
1392,23-09,BLANCO,HB,REC,NaN
1393,23-10,VICUÑA,HB,REC,NaN
1394,23-11,BLANCO,HB,REC,NaN
1395,23-12,BLANCO,HB,REC,NaN
1396,23-13,BLANCO,HB,REC,NaN
1397,23-14,BLANCO,HB,REC,NaN
1398,23-17,ROJO,HB,REC,NaN
1399,23-18,NEGRO,HB,REC,NaN
1400,25-01,VICUÑA,HB,REC,NaN



Filas TIPO-CH: 3


,codigo,descripcion,tipo,titulo
1403,12-12-1-17-34,CH,CH,2/32
1404,14-24-54,CH,HB,NaN
1405,15-18-50,CH,CH,2/32


## 5.2 — Limpiar residuales de descripcion
Valores como TIPO N, MADEJITAS, FT, RET, CH, VARR, SM, STOLL, MADEJAS, MANCHADAS, y combinaciones con -D/-N que quedaron despues de la extraccion de color+color_id.


In [40]:
import re

# Patrones conocidos que deben eliminarse de descripcion
# Orden: mas especificos primero para evitar match parcial
residual_patterns = [
    r'\bTIPO\s*N\b',    # TIPO N
    r'\bCH\b',           # CH
    r'\bSM\b',           # SM
    r'\bSTOLL\b',        # STOLL
    r'\bMAT\s+REC\b',   # MAT REC
    r'\bREC\b',          # REC
    r'-D',                # -D
    r'-N',                # -N
    r'(?<!\w)N(?!\w)',   # N standalone
    r'^\s*-\s*$',        # solo un guion
]

before = dframe['descripcion'].notna() & (dframe['descripcion'] != '')
print(f"Filas con descripcion antes: {before.sum()}\n")

combined_re = '|'.join(residual_patterns)
dframe['descripcion'] = dframe['descripcion'].str.replace(combined_re, '', regex=True)
dframe['descripcion'] = dframe['descripcion'].str.strip()

after = dframe['descripcion'].notna() & (dframe['descripcion'] != '')
print(f"Filas con descripcion despues: {after.sum()}")
print()

if after.any():
    print("Residuales que aun quedan:")
    display(dframe.loc[after, 'descripcion'].value_counts().reset_index())
else:
    print("No quedan residuales en descripcion")
    
dframe.loc[after, ['codigo', 'descripcion', 'color', 'color_id']].head(20)

Filas con descripcion antes: 176

Filas con descripcion despues: 52

Residuales que aun quedan:


,descripcion,count
0,MADEJITAS,11
1,RET,10
2,FT,10
3,BLANCO,5
4,VARR,2
5,VICUÑA,2
6,MADEJAS,2
7,2-32,1
8,GRISS,1
9,MANCH,1


,codigo,descripcion,color,color_id
489,15-26-45,RET,NEGRO,5000
678,22-26-43,FT,BEIGUE HUESO,6003
772,53-25-57,VARR,AVIACION,7042
945,08-26-68,RET,MORADO,7026
1057,16-26-41,FT,VICUÑA,6007
1081,18-26-46,FT,VERDE PACAY,4035
1152,27-24-72,2-32,AZUL MARINO,7013
1226,45-25-34,RET,VICUÑA,6011
1239,48-25-28,RET,MORADO,7026
1242,49-25-66,RET,NEGRO,5000


# 6. Exportar dataset final


In [41]:
OUTPUT = OUTPUT_DIR / 'df-ovillos-cleaned.csv'
dframe.to_csv(OUTPUT, index=False)
print(f"Exportado: {OUTPUT}")
print(f"Filas: {len(dframe)}  |  Columnas: {list(dframe.columns)}")
print(f"\nValores unicos en color: {dframe['color'].nunique()}")
print(f"Valores unicos en color_id: {dframe['color_id'].nunique()}")

Exportado: ../../reference/warehouse/cleaned/df-ovillos-cleaned.csv
Filas: 2132  |  Columnas: ['codigo', 'descripcion', 'unidad', 'Saldoin-Cantidad', 'Saldoin-Valor', 'ent-Cantidad', 'ent-Valor', 'sal-Cantidad', 'sal-Valor', 'sadout-Cantidad', 'saldout-Valor', 'SUB-CATEGORIA', 'tipo', 'estado_mat', 'descripcion_material', 'titulo', 'color', 'color_id']

Valores unicos en color: 187
Valores unicos en color_id: 230


---
## Resumen del pipeline

### Estado actual

| Paso | Estado |
|------|--------|
| 1. Carga y exploracion | Completo |
| 2. Limpieza de `codigo` (tipo, estado_mat) | Completo |
| 3. Extraccion de `titulo` | Completo |
| 4. Separar descripcion en color + codigo | Completo |
| 5. Edge cases y cleanup | Completo |
| 6. Proximos pasos (colores sin codigo, STOLL sin titulo, etc.) | Pendiente |
| 7. Exportar dataset final | Pendiente |

### Archivos generados

| Archivo | Path |
|---------|------|
| Source original | `docs/reference/warehouse/cleaned/ovillos-prod.xlsx` |
| Post-limpieza de codigo | `docs/reference/warehouse/cleaned/df-code-cleaning.csv` |
| Post-extraccion de titulo | `docs/reference/warehouse/cleaned/df-yarn-cleaning.csv` |
| Duplicados | `docs/analysis/warehouse/duplicados_detalle.csv` |
| Dataset final | `docs/reference/warehouse/cleaned/df-ovillos-cleaned.csv` |